# Solving a SAT Puzzle with Grover's Algorithm
This notebook demonstrates Amplitude Amplification and compares it against a classical baseline.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
from IPython.display import display
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from qiskit.circuit.library import PhaseOracle, GroverOperator

print("🚀 Initializing Quantum Puzzle Solver...")

logical_expression = 'x & ~y & z'
winning_state = '101'
oracle = PhaseOracle(logical_expression)
grover_op = GroverOperator(oracle)

print(f"🎯 Target Logical Condition: {logical_expression}")
print(f"🗝️  Target Quantum State: |{winning_state}⟩")

print("\n🎨 Generating Quantum Circuit Diagram...")
display_qc = QuantumCircuit(3)
display_qc.h([0,1,2])
display_qc.barrier()
display_qc.compose(grover_op, inplace=True)
display_qc.barrier()
display(display_qc.draw(output='mpl', style='clifford', scale=1.2, fold=-1))

print("\n⚙️  Running Statevector Simulations (0 to 8 iterations)...")
max_iterations = 8  
iterations = list(range(max_iterations + 1))
success_probabilities = []

for k in iterations:
    qc = QuantumCircuit(3)
    qc.h([0, 1, 2])
    for _ in range(k):
        qc.compose(grover_op, inplace=True)
    state = Statevector(qc)
    prob = state.probabilities_dict().get(winning_state, 0.0)
    success_probabilities.append(prob)

print("\n📊 Generating Quantum Advantage Graph...")
fig, ax = plt.subplots(figsize=(12, 6))
fig.patch.set_facecolor('#f8f9fa')
ax.set_facecolor('#ffffff')

ax.plot(iterations, success_probabilities, marker='o', linestyle='-', 
        color='#4A148C', linewidth=3, markersize=10, 
        label="Quantum Search (Grover)")

classical_prob = 1/8
ax.axhline(y=classical_prob, color='#D32F2F', linestyle='--', linewidth=2.5, 
           label=f"Classical Guessing Baseline ({classical_prob*100:.1f}%)")

practical_k = 2
practical_prob = success_probabilities[practical_k]
ax.annotate(f'Hardware-Optimal Peak\n({practical_prob*100:.1f}%)', 
            xy=(practical_k, practical_prob), 
            xytext=(practical_k, practical_prob - 0.25),
            arrowprops=dict(facecolor='#333333', arrowstyle='wedge,tail_width=0.7', lw=0),
            ha='center', fontsize=12, fontweight='bold',
            bbox=dict(boxstyle="round,pad=0.5", fc="#E1BEE7", ec="#4A148C", lw=2, alpha=0.9))

ax.set_title("Amplitude Amplification: Quantum Search vs Classical Guessing", fontsize=16, fontweight='bold')
ax.set_xlabel("Number of Grover Iterations", fontsize=14)
ax.set_ylabel("Probability of Finding Target", fontsize=14)
ax.grid(True, linestyle='--')
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()
